[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/hamilton-certified/notebooks/day-01-intro-dag-thinking.ipynb#scrollTo=a1b2c3d4)

---
# Day 1 · What is Hamilton & DAG Thinking
**certified-journeys / hamilton-certified** · Day 1 · Foundations

> **Goal for today:** Understand how Hamilton models data pipelines as DAGs from Python functions, install the library, and run your first dataflow.

In [ ]:
%pip install -q sf-hamilton

## Step 1 · The DAG Mental Model

In Hamilton, **every Python function is a node in a DAG**:

| Concept | Hamilton equivalent |
|---|---|
| DAG node | Python function |
| Node name | Function name |
| Input edges | Function parameter names |
| Output | Function return value |
| Execution | Driver.execute(final_outputs) |

Hamilton builds the graph automatically by inspecting function signatures — no explicit wiring needed.

Key rule: **function name = the thing being computed**. Name functions as nouns, not verbs.
- ✅ `age_normalized` — names the output
- ❌ `normalize_age` — describes an action

In [ ]:
# Your first Hamilton module — each function IS a node
import pandas as pd

def spend_mean(spend: pd.Series) -> float:
    """Average spend across all customers."""
    return spend.mean()

def spend_std(spend: pd.Series) -> float:
    """Standard deviation of spend."""
    return spend.std()

def spend_normalized(spend: pd.Series, spend_mean: float, spend_std: float) -> pd.Series:
    """Z-score normalized spend. Depends on spend_mean and spend_std."""
    return (spend - spend_mean) / spend_std

def high_spender(spend_normalized: pd.Series) -> pd.Series:
    """Boolean flag: True if normalized spend > 1 std dev above mean."""
    return spend_normalized > 1.0

### What just happened?
- **`spend_normalized` depends on `spend_mean` and `spend_std`** because its parameters have those exact names — Hamilton wires them automatically.
- **`high_spender` depends on `spend_normalized`** — same mechanism.
- The DAG is: `spend` → `spend_mean`, `spend_std` → `spend_normalized` → `high_spender`.
- No decorators, no explicit graph definition — just Python functions.

## Step 2 · The Driver: Building and Executing the DAG

The **Driver** is Hamilton's execution engine. It:
1. Discovers all functions in a module
2. Builds the dependency graph
3. Executes only the nodes needed to produce your requested outputs

```python
from hamilton import driver
import my_module

dr = driver.Builder().with_modules(my_module).build()
result = dr.execute(['high_spender'], inputs={'spend': pd.Series([10, 50, 200, 30])})
```

The Driver only runs the nodes required to produce `high_spender` — it will not execute any unrelated functions in the module.

In [ ]:
import sys, types
from hamilton import driver

# Package the functions above as a module Hamilton can discover
my_module = types.ModuleType('my_module')
my_module.spend_mean = spend_mean
my_module.spend_std = spend_std
my_module.spend_normalized = spend_normalized
my_module.high_spender = high_spender
sys.modules['my_module'] = my_module

# Build the driver
dr = driver.Builder().with_modules(my_module).build()

# Execute — only request what you need
inputs = {'spend': pd.Series([10.0, 50.0, 200.0, 30.0, 75.0], name='spend')}
result = dr.execute(['spend_normalized', 'high_spender'], inputs=inputs)
print(result)

### What just happened?
- **`driver.Builder().with_modules(my_module).build()`** — discovers all functions and builds the DAG.
- **`dr.execute(['spend_normalized', 'high_spender'], inputs=...)`** — Hamilton traverses the graph backwards from the requested outputs to find the minimum set of nodes to run.
- **`inputs`** supplies the raw data that has no upstream function — the DAG's source nodes.
- Result is a dict keyed by the output names you requested.

## Step 3 · Visualizing the DAG

Hamilton can render the DAG as a graphviz image. This is one of the most useful debugging tools — if the graph looks wrong, your function dependencies are wrong.

```python
dr.display_all_functions('./dag.png')   # saves to file
dr.display_all_functions()             # returns graphviz object (displays in notebook)
```

In Colab, install graphviz first: `!apt-get install -q graphviz`

In [ ]:
# Visualize the full DAG
try:
    graph = dr.display_all_functions()
    display(graph)  # renders inline in Jupyter/Colab
except Exception as e:
    # graphviz may not be installed — print the node list instead
    print('Graphviz not available:', e)
    all_nodes = dr.list_available_variables()
    print('\nDAG nodes:')
    for n in all_nodes:
        print(f'  {n.name}: {n.type}')

### What just happened?
- **`display_all_functions()`** renders every node and edge in the module — not just what's needed for one output.
- **`list_available_variables()`** gives you all node names and their return types — useful for exploration.
- The graph makes dependencies instantly visible — great for code review and documentation.

## Step 4 · Why Hamilton? Comparison with Alternatives

| Approach | Problem Hamilton solves |
|---|---|
| Notebook scripts | No structure, no reuse, hard to test |
| Pandas chained methods | Logic buried in method chains, no lineage |
| Custom pipeline classes | Boilerplate wiring, hard to visualize |
| Airflow/Prefect DAGs | Orchestration overkill for in-process feature logic |
| **Hamilton** | Pure functions, auto-wired, testable, visualizable, swappable backends |

Hamilton is best for: feature engineering, ETL transforms, ML preprocessing — anywhere you have Python data transformations that need to be modular, tested, and reusable.

In [ ]:
# Step 4 demo: the same logic written without Hamilton
# Compare readability and testability

def pipeline_without_hamilton(spend_data):
    spend = pd.Series(spend_data)
    mean = spend.mean()
    std = spend.std()
    normalized = (spend - mean) / std  # can't test this in isolation
    is_high = normalized > 1.0
    return normalized, is_high

# With Hamilton: each step is independently testable
# Example unit test (no Driver needed):
import numpy as np

test_spend = pd.Series([10.0, 50.0, 200.0])
mean = spend_mean(test_spend)
std = spend_std(test_spend)
norm = spend_normalized(test_spend, mean, std)

assert abs(norm.mean()) < 1e-10, 'Normalized series should have mean ~0'
assert abs(norm.std() - 1.0) < 1e-10, 'Normalized series should have std ~1'
print('All assertions passed — Hamilton functions are independently testable.')

### What just happened?
- **Hamilton functions are pure Python** — you can call them directly in tests without a Driver.
- The pipeline version buries logic in a function body; Hamilton surfaces each step as a named, typed node.
- **This is Hamilton's core value**: modularity and testability come for free from the function-per-node model.

In [ ]:
# Challenge: build your own 3-node Hamilton dataflow
# Define three functions where the third depends on the first two.
# Use the Driver to execute the final node.
# Your solution here:

# def node_a(...) -> ...:
#     ...

# def node_b(...) -> ...:
#     ...

# def node_c(node_a: ..., node_b: ...) -> ...:
#     ...

# Then build the Driver and execute node_c
print('Implement your 3-node dataflow above!')

---
## Day 1 key concepts recap

| Concept | What to remember |
|---|---|
| Function = node | Function name is the node name; parameters are its input edges |
| Driver | Discovers functions, builds the graph, executes minimum needed nodes |
| Inputs | Raw data with no upstream function — supplied at execute() time |
| Type annotations | Required — Hamilton uses them for graph validation and visualization |
| Visualization | `dr.display_all_functions()` renders the full DAG |

> **Tip:** Hamilton's core insight: function name = node name, function parameters = edges. Think in nouns (what is being computed), not verbs (how to compute it).

---
## What's next
**Day 2** → Dive deeper into the Driver API: `Builder`, `execute()` vs `raw_execute()`, and how Hamilton resolves the execution order.

Mark Day 1 complete in your [tracker](../index.html).